In [6]:
# 1) 한 번만: 압축 풀기
!unzip /content/추가이미지.zip -d cell_irr/

Archive:  /content/추가이미지.zip
   creating: cell_irr/추가이미지/
  inflating: cell_irr/추가이미지/aug_batch1_001.png  
  inflating: cell_irr/추가이미지/aug_batch1_002.png  
  inflating: cell_irr/추가이미지/aug_batch1_003.png  
  inflating: cell_irr/추가이미지/aug_batch1_004.png  
  inflating: cell_irr/추가이미지/aug_batch1_005.png  
  inflating: cell_irr/추가이미지/aug_batch1_006.png  
  inflating: cell_irr/추가이미지/aug_batch1_007.png  
  inflating: cell_irr/추가이미지/aug_batch1_008.png  
  inflating: cell_irr/추가이미지/aug_batch1_009.png  
  inflating: cell_irr/추가이미지/aug_batch1_010.png  
  inflating: cell_irr/추가이미지/aug_batch1_011.png  
  inflating: cell_irr/추가이미지/aug_batch1_012.png  
  inflating: cell_irr/추가이미지/aug_batch1_013.png  
  inflating: cell_irr/추가이미지/aug_batch1_014.png  
  inflating: cell_irr/추가이미지/aug_batch1_015.png  
  inflating: cell_irr/추가이미지/aug_batch1_016.png  
  inflating: cell_irr/추가이미지/aug_batch1_017.png  
  inflating: cell_irr/추가이미지/aug_batch1_018.png  
  inflating: cell_irr/추가이미지/aug_batch1_019.png  
  inflating

In [8]:
import os
from glob import glob
import cv2
import numpy as np
import zipfile

# ====================================
# 0) 이미지 폴더 경로 설정
# ====================================
BASE_DIR = "/content/cell_irr/추가이미지"   # <- 너 이미지들이 있는 폴더 경로


# ====================================
# 1) 가까운 라인들 묶기
# ====================================
def group_positions(pos, gap=3):
    pos = list(sorted(pos))
    if not pos:
        return []
    groups = []
    cur = [pos[0]]
    for p in pos[1:]:
        if p - cur[-1] <= gap:
            cur.append(p)
        else:
            groups.append(cur)
            cur = [p]
    groups.append(cur)
    return [int(np.mean(g)) for g in groups]


# ====================================
# 2) 테이블 가로/세로 라인 검출 + 이진화(th) 리턴
# ====================================
def detect_grid_lines(img_bgr, debug=False):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    _, th = cv2.threshold(
        gray, 0, 255,
        cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
    )

    h, w = th.shape

    # 가로선
    horiz_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (w // 20, 1))
    horiz = cv2.erode(th, horiz_kernel, iterations=1)
    horiz = cv2.dilate(horiz, horiz_kernel, iterations=1)

    # 세로선
    vert_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, h // 4))
    vert = cv2.erode(th, vert_kernel, iterations=1)
    vert = cv2.dilate(vert, vert_kernel, iterations=1)

    # 가로선 y 좌표
    hp = horiz.sum(axis=1)
    ys = np.where(hp > hp.max() * 0.5)[0]
    h_lines = group_positions(ys, gap=2)

    # 세로선 x 좌표
    vp = vert.sum(axis=0)
    xs = np.where(vp > vp.max() * 0.3)[0]
    v_lines_all = group_positions(xs, gap=3)
    v_lines_all = sorted(v_lines_all)

    if debug:
        print("h_lines:", h_lines)
        print("v_lines_all:", v_lines_all)

    # 최소 개수 체크
    if len(v_lines_all) < 32 or len(h_lines) < 6:
        raise RuntimeError("그리드 라인 검출 실패")

    # ★ 전체 세로라인을 그대로 넘김 (맨 왼쪽 포함)
    v_lines_days = v_lines_all

    return h_lines, v_lines_days, th


# ====================================
# 3) 셀 내부 내용 기반으로 더 타이트하게 조이기
# ====================================
def tighten_box_with_content(th, x1, y1, x2, y2, pad=6):
    h, w = th.shape

    x1 = max(0, min(x1, w - 1))
    x2 = max(0, min(x2, w))
    y1 = max(0, min(y1, h - 1))
    y2 = max(0, min(y2, h))

    if x2 <= x1 or y2 <= y1:
        return x1, y1, x2, y2

    roi = th[y1:y2, x1:x2]
    if roi.size == 0:
        return x1, y1, x2, y2

    row_sum = roi.sum(axis=1)
    col_sum = roi.sum(axis=0)

    row_thr = 255 * 2
    col_thr = 255 * 2

    rows = np.where(row_sum > row_thr)[0]
    cols = np.where(col_sum > col_thr)[0]

    if len(rows) == 0 or len(cols) == 0:
        return x1, y1, x2, y2

    r1, r2 = rows[0], rows[-1]
    c1, c2 = cols[0], cols[-1]

    r1 = max(0, r1 - pad)
    c1 = max(0, c1 - pad)
    r2 = min(roi.shape[0] - 1, r2 + pad)
    c2 = min(roi.shape[1] - 1, c2 + pad)

    return x1 + c1, y1 + r1, x1 + c2 + 1, y1 + r2 + 1


# ====================================
# 4) 4행 × (조 + 30일) 셀 전체 생성
# ====================================
SHRINK_X_RATIO = 0.08
SHRINK_Y_RATIO = 0.12

def generate_shift_cells(img_bgr):
    h_lines, v_lines_days, th = detect_grid_lines(img_bgr)
    boxes = []

    n = len(v_lines_days)

    if n < 3:
        raise RuntimeError(f"세로 라인 너무 적음: {n}")

    col_pairs = []

    # ---- 0열: 조/1조/2조/3조/4조 칸 ----
    col_pairs.append((v_lines_days[0], v_lines_days[1]))

    # ---- 날짜 1~30열 ----
    num_days = min(30, n - 2)

    for c in range(num_days):
        x1 = v_lines_days[1 + c]
        x2 = v_lines_days[2 + c]
        col_pairs.append((x1, x2))

    # ---- 4행 × (1 + num_days) ----
    for r in range(4):
        y1 = h_lines[r + 1]
        y2 = h_lines[r + 2]

        for (x1, x2) in col_pairs:
            w = x2 - x1
            h = y2 - y1

            shrink_x = int(w * SHRINK_X_RATIO)
            shrink_y = int(h * SHRINK_Y_RATIO)

            x1_s = x1 + shrink_x
            x2_s = x2 - shrink_x
            y1_s = y1 + shrink_y
            y2_s = y2 - shrink_y

            x1_t, y1_t, x2_t, y2_t = tighten_box_with_content(
                th, x1_s, y1_s, x2_s, y2_s, pad=6
            )

            boxes.append((x1_t, y1_t, x2_t, y2_t))

    return boxes


# ====================================
# 5) YOLO txt 저장 + 시각화 이미지 저장
# ====================================
CLASS_ID = 0

def save_yolo_labels(image_path, boxes):
    img = cv2.imread(image_path)
    h, w = img.shape[:2]

    base, _ = os.path.splitext(image_path)
    txt_path = base + ".txt"

    with open(txt_path, "w") as f:
        for (x1, y1, x2, y2) in boxes:
            xc = (x1 + x2) / 2.0 / w
            yc = (y1 + y2) / 2.0 / h
            bw = (x2 - x1) / w
            bh = (y2 - y1) / h
            f.write(f"{CLASS_ID} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}\n")

    print("[LABEL] saved:", txt_path)
    return txt_path


def save_visualized(image_path, boxes, out_dir):
    if not os.path.exists(out_dir):
        os.makedirs(out_dir, exist_ok=True)

    img = cv2.imread(image_path)
    for (x1, y1, x2, y2) in boxes:
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 1)

    out_path = os.path.join(out_dir, os.path.basename(image_path))
    cv2.imwrite(out_path, img)
    print("[VIS]   saved:", out_path)


# ====================================
# 6) 모든 이미지 처리
# ====================================
def process_all_images(base_dir=BASE_DIR):
    patterns = ["*.png", "*.jpg", "*.jpeg"]
    image_files = []
    for p in patterns:
        image_files.extend(glob(os.path.join(base_dir, p)))
    image_files = sorted(image_files)

    if not image_files:
        print("⚠️ 이미지 없음:", base_dir)
        return

    vis_dir = os.path.join(base_dir, "labeled_auto_shrink_tight")

    for img_path in image_files:
        print("\n=== processing:", img_path)

        img = cv2.imread(img_path)
        h, w = img.shape[:2]
        if (w, h) != (1570, 195):
            print(f"⚠️ size differs: ({w}, {h})")

        boxes = generate_shift_cells(img)
        save_yolo_labels(img_path, boxes)
        save_visualized(img_path, boxes, vis_dir)

    print("\n✅ done:", vis_dir)


# ====================================
# 7) ZIP로 묶기
# ====================================
def zip_all_labels(base_dir=BASE_DIR):
    txt_files = glob(os.path.join(base_dir, "*.txt"))
    if len(txt_files) == 0:
        print("⚠️ 라벨 없음")
        return None

    zip_path = "/content/labels_all.zip"
    with zipfile.ZipFile(zip_path, 'w') as z:
        for t in txt_files:
            z.write(t, arcname=os.path.basename(t))

    print("📦 zip 생성:", zip_path)
    return zip_path


# ====================================
# 🚀 실행
# ====================================
process_all_images(BASE_DIR)
zip_file = zip_all_labels(BASE_DIR)
zip_file



=== processing: /content/cell_irr/추가이미지/aug_batch1_001.png
⚠️ size differs: (1495, 183)
[LABEL] saved: /content/cell_irr/추가이미지/aug_batch1_001.txt
[VIS]   saved: /content/cell_irr/추가이미지/labeled_auto_shrink_tight/aug_batch1_001.png

=== processing: /content/cell_irr/추가이미지/aug_batch1_002.png
⚠️ size differs: (1064, 140)
[LABEL] saved: /content/cell_irr/추가이미지/aug_batch1_002.txt
[VIS]   saved: /content/cell_irr/추가이미지/labeled_auto_shrink_tight/aug_batch1_002.png

=== processing: /content/cell_irr/추가이미지/aug_batch1_003.png
⚠️ size differs: (1425, 181)
[LABEL] saved: /content/cell_irr/추가이미지/aug_batch1_003.txt
[VIS]   saved: /content/cell_irr/추가이미지/labeled_auto_shrink_tight/aug_batch1_003.png

=== processing: /content/cell_irr/추가이미지/aug_batch1_004.png
⚠️ size differs: (1144, 143)
[LABEL] saved: /content/cell_irr/추가이미지/aug_batch1_004.txt
[VIS]   saved: /content/cell_irr/추가이미지/labeled_auto_shrink_tight/aug_batch1_004.png

=== processing: /content/cell_irr/추가이미지/aug_batch1_005.png
⚠️ size differs:

'/content/labels_all.zip'